# CE497 · Predicting concrete compressive strength
## A first machine-learning and neural-network laboratory

**Engineering question:** Can we predict a concrete specimen's compressive strength from its mixture ingredients and testing age?

We will train and compare a mean baseline, linear regression, a random forest, and a small **PyTorch neural network**. No pretrained model is used: you will see the network learn from data.

**Start:** Open this notebook in Google Colab, connect to a **CPU** runtime, and choose **Runtime → Run all** for a first demonstration. Then return to the experiments and change one setting at a time. No GPU, Google Drive mounting, API key, or live UCI data download is required. The data are bundled inside the notebook as a backup.

**Before running:** Write down two quantities you think matter for strength, and one reason a prediction could be wrong. Revisit your reasoning after seeing the results.

**Learning goals:** Identify features and labels; distinguish training, validation, and testing; explain a loss function and gradient-based weight update; recognize overfitting; evaluate errors in engineering units; and distinguish prediction from physical or causal understanding.

**Scope:** This is a teaching model, not a certified concrete mix-design tool, strength-acceptance test, or structural-design recommendation. The numerical results are specific to this dataset and split.

## 1 · The prediction problem

The [UCI Concrete Compressive Strength dataset](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength), contributed by **I-Cheng Yeh**, contains **1,030 records**, **eight input features**, and one measured target. UCI reports no missing values. Its seven ingredient quantities are masses per cubic metre of concrete, **not percentages**. The target is strength at the **recorded age**, not necessarily 28-day strength. [1]

| Quantity | Role | Units |
|---|---|---|
| Cement, blast-furnace slag, fly ash, water, superplasticizer, coarse aggregate, fine aggregate | Seven mixture features | kg/m³ |
| Age | Eighth feature | days |
| Concrete compressive strength | Target / label | MPa |

For record $i$, the feature vector is $\mathbf{x}_i\in\mathbb{R}^8$ and the measured strength is $y_i$. A model predicts $\hat y_i=f_\theta(\mathbf{x}_i)$. The adjustable model parameters are denoted by $\theta$.

This is **supervised regression**: training examples include numerical target values. Classification, by contrast, predicts categories. We are not solving a concrete constitutive equation or directly simulating hydration.

**Data credit:** Yeh, I. (1998). *Concrete Compressive Strength* [Dataset]. UCI Machine Learning Repository. DOI: [10.24432/C5PK67](https://doi.org/10.24432/C5PK67). Data license: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The packaged CSV preserves the source records; any cleaning is performed explicitly below. [1]

## 2 · Python setup

NumPy handles arrays; pandas handles the data table; Matplotlib draws plots; scikit-learn provides the reference models and data splitting; and PyTorch provides the neural network and automatic differentiation. This cell uses installed packages when available and installs only missing packages. It prints the actual versions for reproducibility.

In [ ]:
import importlib.util
import subprocess
import sys

# Avoid replacing Colab's existing numerical stack.
required = {
    "numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.1,<4",
    "matplotlib": "matplotlib>=3.8,<4", "scipy": "scipy>=1.11,<2",
    "sklearn": "scikit-learn>=1.4,<2",
}
missing = [spec for module, spec in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch>=2.2,<3", "--index-url",
                           "https://download.pytorch.org/whl/cpu"])

import base64, copy, hashlib, io, json, math, platform, random, time, zlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.sparse.csgraph import connected_components
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import radius_neighbors_graph
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

SPLIT_SEED = 42             # Keep the evaluation split fixed during experiments.
MODEL_SEED = 42             # Controls weight initialization and batch shuffling.
DEVICE = torch.device("cpu")
torch.set_num_threads(1)    # Small tabular models do not need many CPU threads.
random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)
torch.manual_seed(MODEL_SEED)

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "torch": torch.__version__,
            "matplotlib": matplotlib.__version__}
print(json.dumps(VERSIONS, indent=2))
print("Device:", DEVICE)

## 3 · Load an archived copy of the data

The loader first looks for the course CSV in the current working folder. When Colab opens a notebook from GitHub, accompanying files may not be present; in that case this cell unpacks an **identical embedded CSV**. It does not contact UCI or GitHub.

A SHA-256 checksum detects changes to this teaching snapshot. The checksum establishes file consistency, not the scientific accuracy of the measurements. The raw archive keeps all records, including repeated records.

The long compressed-data string is infrastructure, not part of the ML algorithm. You can leave this cell collapsed.

In [ ]:
#@title Load the archived UCI dataset (self-contained; no network request)
DATA_SHA256 = "0ad79c3a7c55fff91fb3f21589bef3a52653bb8a498b9b4d11bd17890b5da980"
EMBEDDED_DATA_B64 = (
    "eNqVXUmuJMeR3f9T8ACJhM/DUi1AF9Ci1wRRoARIlFCkGug+fdt7zyIy8/+MCK9ukGKVW3r4YPPkf/72z2+//XH7r3/8/PsfP/3l"
    "P99/+/mXbz/99R8//3r7yz/+96c//f6323///Me377e//uff377/G1B//+Xv/2d/8ed//fz9928//enXX79/+9VAbn/5+2/Pf/zT"
    "r98M5rdfvn/749tPv/zrn//+/u333//+P99++v2P799++/WPv33UEm74/9jSLd3rLQb7i9bbLY1bn/c530DUukG0eB/zI2f+fcG/"
    "wy3Z34fbzOlWZ7mlHm4l3FM/hcqt3kq8h/oR57i3WzTYgm/OBKg+7E8jVftlbjZfueeP1Awslpepmn1s2ni/h/yRh/3h4FP53o/G"
    "7Q+53Us9/ID9odT7qB+lVx7M19/Pe5qXO7GF5nEP/RLQZkwGmD5K6vbH0o+OOd9DtNkwR3h7Lume7VxCudntvdtWv4/Ic3n785ru"
    "0+a33dndh/m0WkOafhuh3Sd3Ne+5feSSbH9f5sHx13aP5ej8J/GltoPjjYP7aKenQSBDzXK1Wh1tuoTLN7uAdgXFz5Z7OjpD7t3Q"
    "Px2M4xJrvh+iLi853mM9OlwswD4Qz1HFELTfSz444A6sbEdYwi3Ue2qH5OGbBKZc4LXfUk9HlNR51TEf4jS/le9tfqSMrerf7+ax"
    "PZV4DmXHEu5hnPIpYHc/XrFoLOXTD2HbxmHS4S2C1MK9tLNL3LnY2ZcAlOcBshFX8j2efWXygmo5gzF8qTy2MiUmvtJOrODsB0hN"
    "LAj33k630m+pAesOJrFFpPtIp3fHc5/3Pg7RiUuxJZ/fH3Agg8YOSASHNu9xXnGLbuu812s5QbhWDlDOJefM15LzNkFLB6smOoT7"
    "rOfSfGDv4XDv5GAmncex9ASEEb0tJIYnhJk93oYtoHGdox9eEicwuGMOxGsckPDv6Ysz1AV+7rrGuerSwTbSMVJBrob7MGoeur9Y"
    "cG4xpmCf4XiP0LNyB46a6oDRHu7RlmL/mob48dZNZtqXbrncTXJGExJGEMlhcbu8+wDsNaXMRJ/BmmQbRrnQ2JrBG6TRu/2pARxi"
    "4jZGt39n3Gn5BBnvWI1NOnOzdWZ9vUE4GR7OfMceo63Nzi8DxzjhnIk3aOoP0DXaDIQbEV+ztQms25+BkEZftsJmkjN27QV/A1pO"
    "2wqhCxgkVMgf2EseXDr+BqAdwplAWqXRQ9cBxY+aIyfExeDcoA9sU05t2/4+/8jHuesUhS2x4mNGwnmHNeaOr1dsKeCb05CMC+g4"
    "dRz9tJsc/gMuwHSKhrM1DKEuXrCliE0CQw0/Ku/ceLHdUMK600h++LgswyUjLMPgfmt1ki3bTnGbE1ux+06+sQl+jckz1txCFWj7"
    "gQWkgXNKwfgJtwWAiC/EEIyX7fiUiM3L05ZsY50nNUD6PEz8oIfOHUHztLuf1MX0aWwlpU+zGTb9yHZy7o8ZOyBxZq+AtpW4QsId"
    "EiatkXCHNrFEwh0iJ12TML8+Fki4w8bpC0TcwUrbChF3SIb5A7tZIeIOjjiviZgb/6GPrxBxd865QMTc/RIR80TnGhFzsWOJiDvU"
    "tx9ZwBoRw1z4gUkvSLjTNrwmYYOb9zDXPpwSjvOShqnILtGwHB/4/AoVG3Sda2SMiQN8B5eE7L6XBUrG59sSKWNOiIQFWgboIj3t"
    "e1ohZyzWbIFretauyo8tYIWkMbFd4xJN68DqChZKy1sUzQCm2rlA1lhvAr9aX8MaYWNzHYtYn/iCuIk017St3f/YqV4SN7TweV9S"
    "smu7tbyqZQN4UaoJNF/T9jbnNW0DkorxJW0bZE9rtI1J55rCu21phbQNFo6ja9IG4KLlsn9/hbIBDOGyQtk4rryG/ADtq5Rdcceg"
    "1gXKxjkQDdbXsEbZwoX5QzNfkDbAIn3qF8QNwLmm9O8He0ncOKnJQMQ1dc/oxLVC3QBu0G6vURGgdUUF15x9gboBme51gboN0g4m"
    "zhXyBixE7A/saYW8DdYG2jV5A3CuGfP791fIG8BjkbxxBnVNxAB0LtrJ2Fr7sWnXKBbQtNvWJ74gWIDlFWPZAI2ltR86qkuCxV3B"
    "9ZOSnJUTlAnCNdMcoA23Pzq4Z8Z36hJoLHQo1BVYBH4KWMYCrLGXNBfntWujNxuxBhH3xDXGiSMpPLBOJEzYWaCn9AoQ+0qImF1C"
    "YlcJYa9LSOyp04l6+XXbkWF4bR9xI6oQSK0VfgIGhI2M4Rev8vYnnNUSsG5szjVo7K4i+LIEbTu0rc3FZXOXJm0K/LgCt28VBL3B"
    "ZSnLJxy2xeMZC4DYXaSb+QoSO4M1cwmITdWlVToi1v6RyKsBGknBRGkgcADDNhk7RWZ2bYvA2JgxrLQGLVKba8DYYLkvzsxLm/cG"
    "YqPEMTlhfzshy4nOAXRrPGfQKW3kFvIKqPYXxgqsdleWYLE5htNW1oC97Ved/aqNHM3sB0MFyppYNr4TkgdNSHULsOInpKMFaIXI"
    "S1+DxhaHmMXKSoSk2RgAVHOKFqhlsTeeCZNNcBp9BDrCIwOFS8DYZcXFLEFjl7yaJWhwT/tTXlxJYMSQkw9GtAk/iNxjQ+sB6ejb"
    "NLu7LgLbNk3fG2kRXOiaVpeC60zIdVlcjAgSAfBIvQVXnXggpqeaEUQEaPS2jcooSByrsLbTTFxZhFeeDiTxGjysCEOvvryewAC6"
    "1jPewONwWiM0YigG5Zu9BPZLLWvQijD6oV9C2zbhCF6EJoUa3dqpGHPqzrUSuVa6Nd45wmjGtbqYbEbyzxKw+FBOa9DCXDDPFWix"
    "2lwXVyI+NKAODRK06QwRmjM93BMnYOYTQlZxKj/gFEy8h2kpp3DKOUmX82E3E7kCV98NW3JchYCZJNABE4gJZxU4PFqU8pnBNS4B"
    "XUVbgMQFGe0vQEpI5IVlckOJ97hZfwHGXqyQ/4O4GqD3TLOTHP2AIivA2JkZX3ENGlcVkYaxBA0iM7DFZfsuXaFqL+DgM2I22eGH"
    "U1lehBYy5r4I7ulhi9DYaFmG5kZNuBGeykHB39bpuiv5ap9T/HIiU+4SEGIhOoacQ2JncNdfAmJTzGm6/rg2BELK0X05VFh7cU6K"
    "KIWpq0Nbsl+WFVDhJmTRNaxycnpegRVeLk3LrVUKIbqxdn11cCKeBBRDslByxi7iXwDG9sRHV6DFWFpdgwZzaRRYSysJzJuFg+EN"
    "eCMCv+6SqV9LwO6PyGvQib46WOQr0NplK4srCVtiKoQ8wZv9bZs0hRtTumlV9Y27tLQCKgu3rYDiEjswegFWVm5dgnU0zQBO+C8T"
    "GinR+5tuFPmBzAgKaqdqndMSqPZWl2AlHNLaGkCDeXFi7s4siQ4FHK4ZemQSHI6x8sKDWBL+p89GyQ7uvQyObdpq4vovtNmY1n8h"
    "9SXWH1iV9m0/STNsPjeYkVB/cDyT6sFoTE6Ck/wSDnKi4CQvIbHBQiF1BfkQFJdf14bAlrr86JOe6YaMR8UaOlzKo1cl0GU6/BZg"
    "bV+FVRgrwDAVBrIVV4BhEEV4upaWEZhUCY5EfY5qHURLZGDFjH8bRchgEkeRk1FXQHFtlSryNaxKL+bSEnB1A06QlTUEptUHcVt6"
    "S+nKKDQkkrScdhv2t0wFi8iQXQDF3mQuXcNKys+lJUgpi0uwD/6ZkLUM4EFtZwgYFiL47BieyY3lXkKCqxjbGgugsoFGXQCV9gIf"
    "3vUCAtOMA8420REl2NLov7RLfd6W3VgfC5CgtYRUkmtQZbKsfF6B8ZFXvm+7akzSToxmyb0GJxDdT8DUwSgX40nkIlChF2B9Z2vA"
    "yjlZW4RSFPLiKrA/014Mutfd/yt7orr7EPUelP1VGxxrsKC1QY16BRo7jDSxVqBxg40ovLQSoOag13p6pDby7zvMDFkViOKY7pa1"
    "yYhqmiVg7LIhoLIErZKvsgYsaTDb4kKwySZLL8sB0xnbrEV5EUmHUu1QmhyBXcGKFWgxzp4XweUggzKxBA42I916bTGBBQlBvr3q"
    "8InwFJQ0rEDavbsnsMscX4H2rS5Cy+c5yyK47rTk1bVgp4xad6DApF5P12ELj7h1i/QHJ2X4BwZOF+HBhTpdtos/QErZrimt/EBJ"
    "QLDXV5cklgSWSw+ijibJsUK9NZLAO7ORZ1JeeycPW/2BdByQ1uovPEsxrv8CG2eZyfqqsHNDC2AHgvh1c/xPpmC026Tax0QMWCt0"
    "v7FOdRVePLktw2+5jqvwQO+sc11ckLhWyzyl6b6fSBUDOTi0VTrjHbNspW25LELLqRPqIrg7h9MiOIysDht0cTGBtW4JmS1NdWUo"
    "M5MLj8ytJfHlBnvhFMjd3ekcSnbj7OdQcmxcrMq9Nqbd1eBFZMX+xuwtU2tr9tzV2r1mruLsja657WljhYGw2rn9WJEqpfyXm2f0"
    "hFsv5C6IiVT7CqZpKPCSMzeDNFi71Qors2v0Gvn9rB9zdGQz1tj2KnqmRAUGQcmxI8RBTb4TVKcxZ6ryH09fthlYVoZ8nspg1LQf"
    "9BQ8Fdr0x8cMVTN0pAJsygyK9liBFnMDVtwSShUN89vY2Ak2Gu7aSUIoD1Vqxc67mXjQUgsiR6UyNngMhjocw5nZ6RTk1+DCSWA0"
    "0/6j5ezfbKg3lPBHaFGeHiYoTWNYNbmOAgzMDIpp7UguAuk9z4bAh60t1P0QvMquzS1N2ug6aM1brV7EpbcWPEE41U8Xlb9eFH4R"
    "I+QjkpwKo6DDljBGcFdlRIgaBDjk2xmJLrmR7J8+/YDgB4aOSOSGGYBNDyR6he0IGR+JqniF+xla1mh72SBSqPpeFor1Sg0rdhON"
    "CwbtG/+wa/fdtuwetIxsOR0LvZ+5akOAse9RBtr5S9LAt6BybRwnPkTX/gCKy19iilAqD2I0asAUdijNMXiyGL/xTgpjHs1zH0Mg"
    "K3aLJZa74OzwsJnMmDPV6qEAMwOw0M8UIjA5GzvOmcYscy1jZ32b+HkHc6fpgTQ76IyGYmAhiYp50g9YZytruPkPmlZjulTs5Dmc"
    "UP8JersVracxD1E6rMkWNH7o1JcZEO9KbITm2Laco2QsOG/mS0CCAvLi6FqBm7Pcuvgzl9498aHiJjHdQNaojIvJ9SD1wwNK6bHy"
    "ypyAT0kYjUKPcn7aIsaWr4VA1VMobiq4j9RlZZp2+mzGvcwvmiL9I9IUi2ycuAnGFj/nXtHJCVKfEyEhlaki0+eL+900kOneek2L"
    "qmIlA85xjUgIoNsdjVVUkpYd5iouyZnINIpFZFKYasRlbFKMBHkgi+jke4gL+ARQFEWt4pPWUusKQuHoSZaXGKX4CCIeSzjli5gL"
    "SKWZ0yJWKd2LnOECrZSnl1axCiKjMbtpDavE9ik9FrFKQiktI5W8iG2dRylmmhZwSkspcRmpFGgecQWpPF13gU0pzkSteQWppDe0"
    "soBUWm/qa0ilk0ZI6AqppPEiZW4Nq2TV9GW5J0836GYVq+QW6OtopYDdWOdVvoe+gFcCLW0Zr7RfJhVf4pXShPoCs1IMq5VFvNIR"
    "praAV4oHIq9tBa+0jhW88pTmuYxYHsIJy5i1JYf9AGp50lxr63JQ22hjXRDKig9tRRIKdqyzLXf3zbIkDD31sy1Iw6DmTsviUEGp"
    "uYJiW/x/VSAqHS7Wj1KMTyPZRVo/Kz5wpjG5AR3yBQyyg9LVPLROP8EEt+42GJLIyTzq1dDOQVJ986VPIBnJRCcgnbHAeA6Sx8WH"
    "uiy8Ewh57EK/gFEztQuYCsv25IBVa9hRwNfpGAAReJHVzOM25MkwvE/lMwxiwE8wIJLxGYbruYB5/hYDxL2efYqXWePZl9gu53xT"
    "HbJ3nO6JrTDm2Xc6CLedfkcXlU8/pMzc2c8+JaQwaZgrjPobXCzwR2RkHNhljh6d6tAB8RUm+YU7jCFFMaSY9ge6gs16NWYFfgR/"
    "xZwPr9AzTIiCSQ+YjszFk3l4U2YMnoCokcY8A1GRZ51nq8luppyAdMrEdAaiM0bHpoQMZQjNGyQiBChgepX7JcGX+w5mCCbDyAKj"
    "eAJpQ8Wbj2lMysSzL4FRnI1jtWzVcvwVLiSeLqQzhXyegaAetzD8H8BhUN9pR2dHiGT5Io+myiiQ5XIMw/5e9Qyis8LqdJLKellj"
    "ammASghjWgoTiMdjMUVRwmMYJruVM4jO1j6nkyjfAm0d4WmegqlMgJ3PR2MYY2rDBRCyv+YpEOkppiuYDG31DIYNT8Bkz2EYqj6D"
    "AQaOq4kUeOvjo+TNsQ+NdWRmKcbpeZ6ZqDGeWwlOepXHpMIUSFP9BIQ+IriVZ36GQMsn1jS43zCdAEzmao2zOWSqo6nbSxO7TC00"
    "ei1mYcLOMQibFyFKcQgx1U4vnYBsXbTyy1IzE89K2pyBDSZOfYIYY7JMmn1oE9ndEUBn0DudnLlaFB5OAD+4EORwkY/g0CFIZ4X8"
    "ySpUH1rP5lBDS+gehzfbmadaDjcDP6hZSJUl6fCNKCTjGrgmYkV4R3wRieEIM+bqPQYdcmIZRhECRKFa+tBsmvlwzsi6EiTGmvDt"
    "6RMos7t6dAeEbYRmCh05M7ws1YwDoFDYfBWJUc9a3YLIL9OyP0FPvoIOJVDf1joOVgBLRznsVyfgERHU9jAWXfqnY7W76KXsdwxU"
    "mn2Lhn09Jm9TCRVT5RcvcDBKO7Rv8XZwk8jk4OjFfNsKezfG5nCR/vTrW5cExJyz3senT08YX73NjWSYXH955LoexI0ZAo3ZYXbY"
    "Tm5QfVom6ZwejxAO2rWMwSewyK4DfWxbSfT5n29FdwIGwczZ9HqILPhv/XHN/fqWqRizvxZuRLfz7l7knewfOhIdz+HBdOZLeAHV"
    "85bZXaJvEyIUiWijuog8z9YmZysbQsSx8mExv7RAhiKtlVvWzYBgOZ2mPppUzH2BY3SaCHaa6mRAxDxGSZBhvqSc3cl7hRtcJyqg"
    "LnfkxF2u0Ff9dcsKW5VfHy2rR/Pc9PgCOzKxXdNeYocLsytko1xF9kj9wqQijb6dDMVHL5iZF25dzkcWUcYlcZE52nQLZyKyaSuY"
    "w00Dca7lh44x87hdFHOFDQfoqntKB6PMNqA+8W4UIZ9OHHo77Bz3aHiyND8eTq4G32Cw7xeOqoXBlH3XheMsPj4GNIzGVgxvBxFe"
    "8xYf74bFbahMvBuWrodktn04P0+OFAa0qXw/jIUn1Tt0Fy8zbuO1eckMesO/HRbTUAnEm2Eq7vnoxzjzSIPv/acHLQNm7L6dvAXP"
    "o6zOMubYhlumTYpK0LeDOLSm1Pt3wzy0LvXp7dw4NSNgE2ehfz5VZEAPNlgpR+NEl8au/+/HGcEvB6NSA2s+GJZBNdPBjUOHRc7g"
    "ltgxlDOTqvu9VKT+fpTZL7yxtHGjKF6UmS9DPhT70bB4Cjxm+3gRo0W2nkvLejis7LnKfG8fz2JbSDYfkg7paNSLze3QwsvkBt1n"
    "lRoOLeH9sMQJ7Md9dqWwZaRHDzWzD+1o2GOuaGW1Te9stG4pZWz9fzDMhuQs/tiGla1t0AMpY7iVno+GsfaAvKK8j2txcBL1IL6K"
    "dKz3w6qdQoUEtZgcKvXaTUOa6KzI1DkaB5Wr9ODJeIbrSFNqLHPXgy7dhDPSkYimm2HHQtwRZSgjiGYaGZK0bircfczGbsrFq2UT"
    "e8Xpk/r8M2jlh2U4o8VpTGp8lSmodriqJL6gBeJBAFhl7AqWVAr9AJ0CReNpTAivVs1PpqXcWmwm1Q+HwYUq46CXC/ImHyt79LKi"
    "SiKO+fHZ2XC4I6ikMCqyHxg0auqCtU9H/WqU6bhTwchojzvA4/IrLz/KAMEdXEyIPbOdCPPt+tPqAszxMaO/OMMA0tXVS9+ap3jk"
    "CRJ4GiDviYIEQk63Maa55xiMvoC8RPPMwDiTBh8eB3Wja2w8DMGHeDWmHk+fxCY71TQy+X4O41mDMFMRw0tPKATXRVYreVY6jbG9"
    "duAnAG+SF/82uIqvblA8iN3LGqMUj3cc6KthgRZ5BRP0jjdPIlM/AN3f8zlSw6R3Gg7hfPox9X7ol0QoiVfzAnHws8jAOiBJyL92"
    "dpri5ONid8S5w49IDqZxeopO8eNy9wLEO1sn+D2pv9ZzbNMrNciaOpmJhjya/l6gE1MtYzm9YJcD45Jr2IegCJ6ReWd64Dy5ur4V"
    "z51eHbB3xAsgPYnVT8hSd4xChZP9yyhBTuMJM5RnAAznAhFkh12zYOB4T5fiXNlfIIYLQGja9Zy98lDPyEqoV/Op0KJ+CPXyBEZK"
    "YlnRQpgJkheEryn85zxY5gQqIk9WJn3GlOezy8ZBXTAGcbu4sEG9ezevaVUxiXbKHOjNQe+1ExrEicKVfXhQHfn3/VKsi7wgsk+O"
    "iuiO/N6LySAgx8o1S5moC1JEVwlL5II0JkuWR/v0MhJ8Isy1pxoGq+L98OaZPxh2pSUfjcup0Q+/jtR/Oltzji/jMPeSs3CzoQ+G"
    "ubpSjobl6+mH43p4bR5Oz9Wp6uTTO2Rsue0Z6TBr3g/D0A3xaNS1wsNxvVkIOfd+HKvb+4s9m00ouehjyiiDh//9sLulUanbXqyu"
    "iQoM/nyoRvjtsBwvSLLfowLNXaBwgMpinG+CmbeeNlXPhOX7YbGDNJ5CDi+TO19B4C6+7A0viPXibtqn0fo86jGIR22JCw04GVq0"
    "S8XBtcPh7Jb2wTDTTszQPpq8+NIPxj3R5umJVHGfx/Q1AWsOhm16E8dGcQfjKvHJh7+fKG0C1h3Nb/dlrK2lww/Ani90ZNZtg/T+"
    "4Fp6FVrhybi3o2rfV4+GxW/gv3k7rKbKcxwMy6jL9ejbcvf1raZtwyoUAHb45CAZ+8EgOdXRIPW+dDgv/cbzaNg91vUTl5pI2Um+"
    "6nkwyMfmIhn425cUvRzp8Oc0f7Cw9EqFHeEBOvOPxjxw8OW39n8IGXjAoB8R8E6iu7dbJBzAmGrxWFk7GtaZDTZt+TLe/JnEcTT6"
    "YG15vPCPOA44px8qnJut7m7A96MK5qLz2Ptx+Se+XCmK/F7u5O2N0QbBk5ZvR3Us5QiVJA9RvvoeWXbSfT8sWwLk835cOhVc0u/H"
    "/elWvOp5gKxte7s1h/SJgEdkybsv8P3woJJs+kDd3bvzM29rDBge8GZZi+lwnO9dsrvREXPXK5HxkL3yWT52YEuvmLdTTWYrg/fD"
    "iuq1+ZUswhNZvMtjoo+YOUzw07wdlSeojoNh5TchIDFe1nbALOpbZvHY13tmcbXtT69/Brz+KXKcR6Oe0pIOhifjKEjueTtM7tv5"
    "AvbbcYaIBg71AOf4vOM4GuVjfXYuRwhbcOaH+Kxc23k4Ptkzph9+HdiKZwgMI9IXgoKdqRq3fjC8s9GDX4/gtXfvyVV88jUr57MO"
    "yPfa3quQTGEzLlr943ohdczGfzqN/XgwqteiEKOCFRxgDTB1GEq5mVWqyc70BDDrHa+0lsEUTlOuB0Iu6gaA6KDZkMhbLHgyxKZB"
    "sTPsWpQxk6jwQomZkMBx+5ARg53GVEl2mo7bqIapStONeIOjISKL1y+2OnS0N5ARFwtMukgYPLthl9RqfCrSK8Ur2VXzDrv4Noqo"
    "iCljmc8MeJ+EG07E5Era3NgA6FtIUi4fIGreCszhBmRvhKmCdn7IrNy89RVIqrtBR4U8nUWyAibZyW7PFrjBis0EbTdwK/6wQeEL"
    "msGb4NpCGgHQTVW9CRCtgr8keiCNtTp2GnhOpCffKxy7aOGAsczSGGQGD8QBFfVGFR8MoMFmDjF0pnXbidg9eBF6lTCGtdiKn0ll"
    "lwRXY1CHgxcS2xZSm3RCJLbRpDpeaWWxEF4VOLETA2CF7o+xDTFHI4XeGH83kZ0fZm6+75XwVeE3tsFKOPXpAfwA9yWqNbLHVSM7"
    "ViM/fStEzN5kYLBXGAkSJ2JT9Bg2Z2HhiTA0SUbdpJtEHRktpxz9CSKSTUPjZUOR7k2T0EBt4D4iyxHgmxFhza34FBnNSKRvTSVL"
    "N6Xs9xn8kUIoKepyUJjYw2UitbiP7XFkZHDZTWORzdOrbR0QOFuEKpMzxTJd4tg6Kh4y2pqRIcrQiqMoIsmIIhd0xW6OQUwnU+lT"
    "ZpMNknbdg8WDVRVJ7buG7iUKO3Ymgnq+wvdvCJPsmHBg0xBDC1WnwsE2MzwO9ucOY+MPUoked4IjJx7DENt1YZOe7CNiA40tR5oC"
    "5nPzs/IpAcQDw0ZywCDbbhub8C90TRl9btFtHEgFqnbfDDAZCBqp7rHNhLarhiCqF5M8IHcQ4Y6IxIHN1EEKHLhHZmcV3i6YUNrO"
    "LImPgcmIovhBo+2waedQQ/GUt0rxgBIhkqcit2hzeiMwYOcI/6/RBA/deJLzU73EkGPe79ZWuDGp6RwXXKxuUX5b5ZRG02bfEvSw"
    "ExQVSLqx30FCrczGTEEJs233BiZi1wP/jNAYynqSWzuK4fJRDnjQ56bNBy0TxTjBlwksB4Px/iWSHZTr5IUDT4HZZ5pnU0RxY7Ug"
    "ebdVJEA7kmZ2oSFB4AUpFPdlPw1kdSFeK88s7iSzf/yYW7Yp/cDcByQB0YustKex5emXN0t52k5icyHI2iCWzCYwWqz7IxHzqOQe"
    "AIH0nDtzmMmZQ3tsOTw2HFiWIokbDJFZ9LbTS5UgzFuyYGfq7k39eyjW8RoZnNWEGCQoF0DVi0Vxz8bndLuRFciJBgDfeedCJ0LZ"
    "6rSCZ9Z0qLA8ZyKqdcnSnRTQwQMiH4+04X/xtpY62JTsiQkItTR6XUELsUeyqby7F4oyAXHgLfqBUN6i0HVnhl7Qkjb20Knk9rm1"
    "wpt4pKvt6R+Gp2CpUuQMDZEuquNGh5pBVJ+UG8kPA+cZyP9cFieZ5E2OH1gImeoLGW1mpSnQg2rBnoQhXzJAdDMA2JvtoA9XJNW6"
    "ovSVahPTIT7R9TCeO9qGpYhwZU7RZLXisFrcVoGtmOqIMbi5++M89+5Nmc0sIDoeqD6J6rZd58jqo/UgbTCOFyxVeRAYjBuRn08d"
    "Sy06MYQAYWJ33luJu5OFhZ5OUJBOuJiKmBledtv0JHa2Q29GPLImBWYW5Nk3d+QiEPHQCSDJsqNpcRFGhe4Lxe0sRna/jr341Ujq"
    "h60kMLH+HpUawKDgMmxCkWr7iWkv3qcJjH07sbHl46YNDbtU7aEWSnL7xMDsr+gG7hfGAHvmg4cJ3pBYZb6p870Vx8HQv6DYEwLN"
    "rRAEhYN88g3bANMvW1Bg0Kyg6gJRESUkoanlTRSDU2YJ0Ckm18htXfNI7FbPm4CHkTytkzHErVNZoquHH2Jnm8LjiMrY2mycAG8B"
    "A26AQeusQJGO5Yat9QduDkyq6tZEltMopKXtpa7KtHbaUtTW0IgL4b+dS7EdKWR6gsiHkGah2Ch433dv6DuEzZWXDxnDB652nU2N"
    "82hu8dFO8OXBWWDQJl9LZocdKg+chimJ4NxNmmFiYIKSslGfTzET0/K+oQRuB2WapD0ku/G4HfqVbWhQmPLKdxHRIY+vYesdUL6C"
    "B0r3J4V2beP+pG+gFhDIh0ijqWDeEV5aB5uGMleaHIotBaIyt6lG33cNBAYf37lD4qK0EMQL2TmMVbueUZLUtlq6CH+vdrP41YhM"
    "u+hjb7gnrcQzvRVmRa41I9GFneb34qlBZZ5b24mfWzOxjsBnepIfeAnTHT987BH939S3DfqKP9e3ayzsOVRPj8s1l8wOClmvg2V/"
    "G5M97Tcd5l5etBhclesxbGSAXw5FwLdWmdJnEvtdY8Xspx2jN+c1tQZHlrZHhjbV5s2aX85hV3CUF5+betPxjSx1lsCFjRyeEzxM"
    "2VFuvN7IhB4R2P4YD3Si7+bWcgePi1U1B/XuDp3vptkBo1bSOMz9Rf/hLQgF+HXiK/UgbC4/cv2kCzk6ss8kiy+pEXUif9LluVb0"
    "KA/IzB9QCRq1Izw18aofKdtksLHm1Ds3LF5H0L03X3JmVrB0pc53bvn+Dz7LV195xX1sz3Q+603si8VT4fuHgz32SUgl78mdTG5V"
    "Iw6V9d2ph3DR+AEra7Ca+axR4Y9oAiu8794e1HCObTH8/ZjKty2pXEV/V6oNoifs/UBqKvf0ombttRhM2cACBhB1RE/cgsIF70YR"
    "dubsDx4Ovdo6+fDsILq+Kl/eT5t9S7x9K+uxGlsyP6lh6GKstiBOWBO0S5RG2hWUoXt+Vsr0UOYUgRNJiHtSzgQK9Sw5e6Npj0Uc"
    "szdX1d4zw8FevCCu/qy2Od43PQ+lvsuuvHl/1F19Y4cUrGyoJ7X6vPB1T1wTqsq+KnNAaJaPgrRcpbPVNFYgPal1L1SImOZXKtz0"
    "Oy7VqeDwwjddj/eW2eadOT+icHZugY0PvW+vTcLjGRunGezhM9RinBXU1AC9DqcyMYo6YNNTu6CVJj0QFNu2hyD07M30WYO34GRf"
    "YPyw4mietML+VjR8ZuO7dnhPnj+DjaqiCwoVtOzJ3JgtG/tJV/R22+pni3Ng0ZHpjN5frIlxsKvQ1mq9M233/tAe7/VFfxSu8YZN"
    "T7mzZsH1SF/ypkmme990yXecdlMo2U6IzyYlCpwed87CrCGURL2ol+9J5BPSb6omRlki1vmYT2N7I1c59+K3sZVPBTZyomJARHfn"
    "XPdy+he/N+UZxVPnAwOD0okoh8Ztzz5wPY9WdqpCweb+ljpus/ER8IdHnD2Y2KlXVX/ECX5AvnHnXe4dZ0siHnLMFBKNuDfYEakw"
    "H609ecqDXq/ghtWguzG5Vi95A8Va3XK3B0J+aoq1PT4taaemspQdY+MGmw+d26XiQQWSLyipqJze9P2NLfnTt+fNySb0bBa5SEcq"
    "4eZZbxSyjSe+e9fZjpmElZ9aHriPnRKax+F+djViwlk10tEnb7sXcjaxOnjc2az22edetsc/3PggVVPlC3XPfqTv3Z8fF72ykZW6"
    "zkwmy/ZN13A/fHxA8XVF78YV1K3FK+nzs1eeFJvV5qo5JWZ55yEmqoPvHnrqGsXV1eDv437y1KdXTz3mA5CqDPmG4MNj39nLLHlL"
    "LqIDEl4pWfkexS5S4L/XUxxRL97rdRtyL7YDzuz8kR4UAxPFe1RXTj3io3t+4l0n72G4efbJgctW4qp3wwUOD3/fiVc+fnJSvQhS"
    "mDaY1GMavK7DnnL4yhK2Bz/nA5GFr12619+Flfv9u7cJB/GKfbCVMXWgnrf2klsMgM91YGgSSYY0JGSqsvXzdJUfUg01WexK15z5"
    "FjWS4cE8BQaQG1T94fvop7yFB/wduvI5RNC8V/90ns7bQIN+W3LdldzMOCibog4XxhTBfF6SNuhDn8sfG6olZnZGZqe6HcH3Gl9j"
    "CN5qjwBQkubGGNkJY7hauccTdASJZKoXBfjEELvP9SdGjZZ+mdTUlLlNNMiN1zIorRop5TXSwLZ7HmvAmdl9dDbt8R0WvRw/vWUB"
    "42LS94UWTC8v+7NeyLtufA+B3tR79fepcU2DymOfzuY8DlGn8l9dF2Z5qqIRoMgx9o7gpSkiUV1xHlnHDFucoiK1XW1FQaIe12B0"
    "gm8rT+qsczs66t3PcQp/lKTet1jFncFp/CLm3RxmxEKtEPVGJbk0LUUSykwyPjZFjRvc3lqVkdlEfpNmTh9O2zDLEbjx1ougoeHm"
    "K9aNOFFhA3tX5BnW+H82qihV"
)

def canonical_bytes(data: bytes) -> bytes:
    """Normalize Windows line endings without changing any field values."""
    return data.replace(b"\r\n", b"\n")

def validate_snapshot(data: bytes) -> bytes:
    data = canonical_bytes(data)
    if hashlib.sha256(data).hexdigest() != DATA_SHA256:
        raise ValueError("CSV checksum mismatch. Restore the supplied course CSV; "
                         "do not silently substitute a different dataset.")
    return data

def load_snapshot() -> tuple[bytes, str]:
    candidates = [Path("data/concrete_strength.csv"),
                  Path("concrete_strength/data/concrete_strength.csv"),
                  Path("concrete_strength.csv")]
    for path in candidates:
        if path.is_file():
            return validate_snapshot(path.read_bytes()), str(path)
    data = validate_snapshot(zlib.decompress(base64.b64decode(EMBEDDED_DATA_B64)))
    path = Path("data/concrete_strength.csv")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return data, "embedded backup → " + str(path)

csv_bytes, data_source = load_snapshot()
raw = pd.read_csv(io.BytesIO(csv_bytes))
ORIGINAL_COLUMNS = ["Cement", "Blast Furnace Slag", "Fly Ash", "Water",
                    "Superplasticizer", "Coarse Aggregate", "Fine Aggregate",
                    "Age", "Concrete compressive strength"]
assert list(raw.columns) == ORIGINAL_COLUMNS
assert raw.shape == (1030, 9)
assert np.isfinite(raw.to_numpy(dtype=float)).all()

FEATURES = ["cement", "slag", "fly_ash", "water", "superplasticizer",
            "coarse_aggregate", "fine_aggregate", "age_days"]
TARGET = "strength_mpa"
raw.columns = FEATURES + [TARGET]
print("Data source:", data_source)
print("SHA-256:", DATA_SHA256)
print(f"Raw records: {len(raw):,}; features: {len(FEATURES)}")
print("Missing values:", int(raw.isna().sum().sum()))
print("Exact duplicate records:", int(raw.duplicated().sum()))

## 4 · Split first: training, validation, and test sets

| Set | What it is for | What must not happen |
|---|---|---|
| Training | Fit preprocessing and model parameters | Do not fit using held-out labels. |
| Validation | Choose settings, model, and stopping epoch | Do not call this the final independent test. |
| Test | Evaluate the frozen choices at the end | Do not tune a model after inspecting its test errors. |

**A dataset-specific trap:** Many records have the same mixture at different ages. There are also very similar ingredient records. Randomly splitting individual rows can place closely related mixtures in both training and test sets. That addresses a different, easier question than predicting strength for a held-out mixture family.

For this lesson we remove **exact duplicate rows** once, keeping the first occurrence. We retain identical inputs with different measured strengths. Without specimen identifiers we cannot tell whether every exact repeat is a duplicate entry or an independent replicate; this is a documented teaching choice, not a claim that those observations are invalid.

We put similar recipes into **recipe families**, excluding both age and strength from this grouping. Two recipes are linked when all seven ingredient differences are at most **1 kg/m³**, and connected recipes stay together. This threshold is a conservative course-design choice, not a UCI-supplied identifier. Transitive links can group recipes farther apart than the threshold. It also can merge genuinely distinct mixtures.

We assign approximately **60/20/20% of families** to training/validation/test. The percentages of **rows** need not be exact. This prevents splitting these recipe families, but it does **not** guarantee independence between laboratories, source studies, or batches, whose identifiers are not available here. [2,3]

**Pause:** Why must `age_days` remain a model input but be excluded from the recipe-family definition?

In [ ]:
keep = ~raw.duplicated()
source_row_ids = np.flatnonzero(keep.to_numpy()) + 1  # Original UCI data-row numbers.
df = raw.loc[keep].reset_index(drop=True).copy()
X = df[FEATURES].copy()
y = df[TARGET].to_numpy(dtype=np.float64)

# Chebyshev distance = the largest absolute ingredient difference.
# Every ingredient has the same unit, kg/m^3. Neither age nor strength is used.
RECIPE_TOLERANCE = 1.0
recipe_graph = radius_neighbors_graph(
    X[FEATURES[:-1]].to_numpy(), radius=RECIPE_TOLERANCE,
    metric="chebyshev", mode="connectivity", include_self=True, n_jobs=1)
n_families, family_id = connected_components(recipe_graph, directed=False)

outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SPLIT_SEED)
development_idx, test_idx = next(outer.split(X, groups=family_id))
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SPLIT_SEED + 1)
a, b = next(inner.split(X.iloc[development_idx], groups=family_id[development_idx]))
train_idx, val_idx = development_idx[a], development_idx[b]

assert not (set(family_id[train_idx]) & set(family_id[val_idx]))
assert not (set(family_id[train_idx]) & set(family_id[test_idx]))
assert not (set(family_id[val_idx]) & set(family_id[test_idx]))
assert sorted(np.r_[train_idx, val_idx, test_idx]) == list(range(len(df)))

X_train, X_val, X_test = (X.iloc[i].copy() for i in (train_idx, val_idx, test_idx))
y_train, y_val, y_test = (y[i].copy() for i in (train_idx, val_idx, test_idx))
split_summary = pd.DataFrame([
    {"set": name, "rows": len(idx), "row_fraction": len(idx) / len(df),
     "recipe_families": len(np.unique(family_id[idx]))}
    for name, idx in [("training", train_idx), ("validation", val_idx), ("test", test_idx)]
]).set_index("set")
print(f"Cleaned records: {len(df):,}; recipe families: {n_families}")
display(split_summary.round(3))
print("Family overlap between every pair of sets: zero.")

## 5 · Look at the training data and make an engineering hypothesis

Explore the **training** set, not the test set. The plots below show strength against age and against the mass ratio $w/b=\mathrm{water}/(\mathrm{cement}+\mathrm{slag}+\mathrm{fly\ ash})$.

The ratio is an exploratory derived feature; the default models still use the original eight inputs. A scatter plot combines many different mixtures, so a trend is not a controlled experiment. Even a useful empirical relationship is not proof of causation.

**Discuss:** Is a single straight line likely to describe the age–strength relationship across all mixtures? What information about specimens or curing conditions is missing from these eight inputs?

In [ ]:
train_table = X_train.copy()
train_table[TARGET] = y_train
display(train_table.head(5).round(2))
display(train_table.describe().T[["min", "mean", "max"]].round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_train["age_days"], y_train, s=18, alpha=0.45)
ax.set_xscale("log")
ax.set(xlabel="Testing age (days; logarithmic axis)", ylabel="Measured strength (MPa)",
       title="Training data: age alone does not determine strength")
fig.tight_layout()
plt.show()

binder = X_train[["cement", "slag", "fly_ash"]].sum(axis=1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_train["water"] / binder, y_train, s=18, alpha=0.45)
ax.set(xlabel="Water / (cement + slag + fly ash), by mass",
       ylabel="Measured strength (MPa)", title="Training data: an engineering feature")
fig.tight_layout()
plt.show()

## 6 · Baselines and error metrics

**Mean baseline:** Predict the training-set mean for every specimen. A useful model should improve on a simple reference.

**Linear regression:** Learn an intercept and eight coefficients, $\hat y=b+\sum_{j=1}^8w_jx_j$. Its response is linear in these original inputs. We standardize its inputs inside a pipeline.

**Random forest:** Fit many decision trees, each making threshold-based splits, and average their predictions. This is a nonlinear reference model; it does not use the neural-network gradient update shown later. It does not need feature scaling. Here its settings are fixed before testing. [4]

For evaluation on $N$ held-out examples,
$$\mathrm{MAE}=\frac1N\sum_i|\hat y_i-y_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac1N\sum_i(\hat y_i-y_i)^2}.$$

Both are measured in **MPa**. RMSE gives more influence to large errors. $R^2=1-\sum_i(y_i-\hat y_i)^2/\sum_i(y_i-\bar y_{\mathrm{eval}})^2$ is dimensionless: 1 is perfect, 0 matches the evaluated set's mean reference, and negative values are possible. **$R^2$ is not classification accuracy or a percentage of correct predictions.** [5]

The mean baseline uses the **training** mean, so its held-out $R^2$ need not be exactly zero. Fit scalers only to training data. [2]

In [ ]:
def metrics(truth, prediction) -> dict:
    truth = np.asarray(truth, dtype=float).reshape(-1)
    prediction = np.asarray(prediction, dtype=float).reshape(-1)
    if truth.shape != prediction.shape or not np.isfinite(prediction).all():
        raise ValueError("Predictions must be finite and match the target shape.")
    return {"MAE_MPa": mean_absolute_error(truth, prediction),
            "RMSE_MPa": math.sqrt(mean_squared_error(truth, prediction)),
            "R2": r2_score(truth, prediction)}

reference_models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2, random_state=MODEL_SEED, n_jobs=1),
}
for name, estimator in reference_models.items():
    estimator.fit(X_train, y_train)  # Never fit on validation or test observations.

validation_baselines = pd.DataFrame([
    {"model": name, **metrics(y_val, estimator.predict(X_val))}
    for name, estimator in reference_models.items()
]).set_index("model")
display(validation_baselines.round(3))
print("Validation results only. The test set has not been scored.")

## 7 · What does the neural network learn?

This **multilayer perceptron (MLP)** takes the eight input numbers and learns nonlinear combinations of them:

```text
(batch, 8) → Linear(8, 64) → ReLU → Linear(64, 32) → ReLU → Linear(32, 1)
 features        hidden features       hidden features       one output
```

For one specimen, $\mathbf h_1=\mathrm{ReLU}(W_1\tilde{\mathbf x}+\mathbf b_1)$, $\mathbf h_2=\mathrm{ReLU}(W_2\mathbf h_1+\mathbf b_2)$, and $\hat z=W_3\mathbf h_2+b_3$. ReLU means $\max(0,u)$ and introduces nonlinearity. All entries of the **weight matrices and bias vectors** are learned. The hidden values are intermediate features, not known physical variables or 64 separate strength predictions. [6,7]

The default network has $(8\times64+64)+(64\times32+32)+(32\times1+1)=\mathbf{2,689}$ trainable numbers. Without nonlinear activations, stacking these linear layers would still give a single linear mapping.

### Why scale inputs and the target?

For each feature, $\tilde x_j=(x_j-\mu_j)/s_j$, using **training-only** means and standard deviations. We also standardize the target, $z=(y-\mu_y)/s_y$, to keep the optimization scale convenient. After prediction we convert back using $\hat y=\mu_y+s_y\hat z$.

Scaling does not add observations or teach the network physics. The final output layer is linear, with no ReLU: standardized target values can legitimately be negative even when physical strength is positive. [2,6]

In [ ]:
x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
y_mean = float(y_scaler.mean_[0])
y_scale = float(y_scaler.scale_[0])

# Column-vector targets prevent accidental broadcasting from (batch, 1) to (batch, batch).
Xt = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32, device=DEVICE)
Xv = torch.tensor(x_scaler.transform(X_val), dtype=torch.float32, device=DEVICE)
yt = torch.tensor(y_scaler.transform(y_train.reshape(-1, 1)),
                  dtype=torch.float32, device=DEVICE)
yv = torch.tensor(y_scaler.transform(y_val.reshape(-1, 1)),
                  dtype=torch.float32, device=DEVICE)
assert Xt.shape[1] == 8 and yt.shape == (len(X_train), 1)
print("Training features:", tuple(Xt.shape), "; training labels:", tuple(yt.shape))
print(f"Training target mean = {y_mean:.2f} MPa; scale = {y_scale:.2f} MPa")
print("Example first training feature vector (standardized):", Xt[0].numpy().round(2))

In [ ]:
#@title Neural-network settings — change one setting at a time
HIDDEN_1 = 64 #@param [8,16,32,64,128,256] {type:"raw"}
HIDDEN_2 = 32 #@param [8,16,32,64,128] {type:"raw"}
LEARNING_RATE = 0.001 #@param {type:"number"}
BATCH_SIZE = 64 #@param [16,32,64,128,256] {type:"raw"}
MAX_EPOCHS = 400 #@param {type:"integer"}
PATIENCE = 60 #@param {type:"integer"}
WEIGHT_DECAY = 0.0001 #@param {type:"number"}

if min(HIDDEN_1, HIDDEN_2, BATCH_SIZE, MAX_EPOCHS, PATIENCE) < 1:
    raise ValueError("Widths, batch size, epochs, and patience must be positive.")
if LEARNING_RATE <= 0 or WEIGHT_DECAY < 0:
    raise ValueError("Learning rate must be positive; weight decay must be nonnegative.")

class StrengthNet(nn.Module):
    def __init__(self, width1=64, width2=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8, width1), nn.ReLU(),
            nn.Linear(width1, width2), nn.ReLU(),
            nn.Linear(width2, 1))

    def forward(self, x):
        return self.network(x)

torch.manual_seed(MODEL_SEED)
model = StrengthNet(HIDDEN_1, HIDDEN_2).to(DEVICE)
print(model)
parameter_table = pd.DataFrame([
    {"parameter": name, "shape": str(tuple(p.shape)), "numbers": p.numel()}
    for name, p in model.named_parameters()
])
display(parameter_table)
print("Total trainable parameters:", sum(p.numel() for p in model.parameters()))

## 8 · Loss, backpropagation, and one update

For a mini-batch of $B$ specimens, the data-fitting loss is
$$\mathcal L_{\mathrm{batch}}=\frac1B\sum_{i=1}^B(\hat z_i-z_i)^2.$$

This loss is **dimensionless** because the target is standardized. Multiplying it by $s_y^2$ gives the corresponding mean squared error in MPa². The labels are the measured strengths; the eight features are inputs, not the quantities being minimized.

A gradient tells us how changing a parameter would locally change the loss. Plain gradient descent would use $\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L$. Here **Adam** uses running gradient statistics to adapt the update. `WEIGHT_DECAY` adds an L2-style penalty through the optimizer; the displayed data-loss curves do not include that penalty. [6,8]

```python
optimizer.zero_grad()  # Clear gradients left over from the previous batch.
prediction = model(xb) # Forward pass: predict each specimen in this batch.
loss = loss_fn(prediction, yb)
loss.backward()       # Compute parameter gradients; this does NOT update weights.
optimizer.step()      # Update the weights and biases ONCE for this batch.
```

**One epoch** visits all training records. With $N$ records and batch size $B$, an epoch has $\lceil N/B\rceil$ updates. Computing a loss alone does not change the model. Validation computes predictions without parameter updates. [6]

The next cell performs one update on a **throwaway copy**. It shows gradients and a real parameter change without altering the network that we will train below. A single step is not guaranteed to reduce every loss.

In [ ]:
demo = copy.deepcopy(model)
demo_optimizer = torch.optim.Adam(demo.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()
xb, yb = Xt[:BATCH_SIZE], yt[:BATCH_SIZE]
weights_before = demo.network[0].weight.detach().clone()
demo_optimizer.zero_grad()
prediction = demo(xb)
assert prediction.shape == yb.shape
loss = loss_fn(prediction, yb)
loss.backward()
print(f"Mini-batch MSE before update: {loss.item():.4f} (standardized target)")
print(f"First weight-matrix gradient norm: {demo.network[0].weight.grad.norm().item():.4f}")
demo_optimizer.step()
change = (demo.network[0].weight.detach() - weights_before).abs().max().item()
print(f"Largest absolute change in a first-layer weight: {change:.6f}")
print(f"One epoch will contain {math.ceil(len(Xt) / BATCH_SIZE)} updates.")
del demo, demo_optimizer  # The main model is still at its initial weights.

## 9 · Train, validate, and stop before overfitting

The function below contains the entire training loop. Training batches change the weights. At the end of each epoch we compute training and validation errors with the **same current weights**, without updates.

We save the weights with the lowest validation MSE and restore them at the end. **Early stopping** ends training after `PATIENCE` epochs without a new best validation score. Validation therefore influences which fitted model we keep, but its labels are not used for gradient steps. The test set is absent from this function.

Widths, learning rate, batch size, weight decay, and stopping rules are **hyperparameters** chosen by us. Weights and biases are **parameters** learned by optimization. A low training error by itself is not evidence of useful prediction.

In [ ]:
def train_network(net, *, learning_rate, batch_size, max_epochs, patience, weight_decay):
    generator = torch.Generator().manual_seed(MODEL_SEED)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size,
                        shuffle=True, generator=generator, num_workers=0)
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate,
                                 weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    best_loss, best_epoch, stale_epochs = float("inf"), 0, 0
    best_state = None
    history = []
    updates = 0
    started = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        net.train()
        for xb, yb in loader:
            optimizer.zero_grad()       # 1. Reset gradients.
            prediction = net(xb)        # 2. Predict the batch.
            assert prediction.shape == yb.shape
            loss = loss_fn(prediction, yb)  # 3. Compare with measured labels.
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite loss. Try a smaller learning rate.")
            loss.backward()             # 4. Backpropagate gradients.
            optimizer.step()            # 5. One update per mini-batch.
            updates += 1

        net.eval()
        with torch.no_grad():
            train_mse = loss_fn(net(Xt), yt).item()
            val_mse = loss_fn(net(Xv), yv).item()
        if not np.isfinite([train_mse, val_mse]).all():
            raise RuntimeError("Nonfinite evaluation loss. Reduce the learning rate.")
        history.append({"epoch": epoch,
                        "train_RMSE_MPa": math.sqrt(train_mse) * y_scale,
                        "val_RMSE_MPa": math.sqrt(val_mse) * y_scale})

        if val_mse < best_loss:
            best_loss, best_epoch, stale_epochs = val_mse, epoch, 0
            best_state = copy.deepcopy(net.state_dict())
        else:
            stale_epochs += 1
        if epoch == 1 or epoch % 50 == 0:
            print(f"Epoch {epoch:3d}: train RMSE {history[-1]['train_RMSE_MPa']:.2f} MPa; "
                  f"validation RMSE {history[-1]['val_RMSE_MPa']:.2f} MPa")
        if stale_epochs >= patience:
            print(f"Early stopping after epoch {epoch}.")
            break

    assert best_state is not None
    net.load_state_dict(best_state)
    net.eval()
    elapsed = time.perf_counter() - started
    print(f"Restored epoch {best_epoch}; {updates:,} updates; training time {elapsed:.1f} s.")
    return net, pd.DataFrame(history), best_epoch, updates, elapsed

# Reinitialize on every rerun: do not accidentally continue from trained weights.
torch.manual_seed(MODEL_SEED)
model = StrengthNet(HIDDEN_1, HIDDEN_2).to(DEVICE)
model, history, best_epoch, update_count, training_seconds = train_network(
    model, learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, weight_decay=WEIGHT_DECAY)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["epoch"], history["train_RMSE_MPa"], label="Training")
ax.plot(history["epoch"], history["val_RMSE_MPa"], label="Validation")
ax.axvline(best_epoch, linestyle="--", label=f"Restored epoch {best_epoch}")
ax.set(xlabel="Epoch", ylabel="RMSE (MPa)", title="Learning curves: fit versus generalization")
ax.legend()
fig.tight_layout()
plt.show()

## 10 · Compare models using validation data

We now compare all four models and choose the lowest **validation RMSE**, before inspecting test performance. A neural network is not automatically the best model for a dataset. The model ranking is an empirical result of this particular experiment, not a universal ordering.

**Read the learning curves:** Falling training error with rising validation error is evidence of overfitting. If both remain high, the model or training procedure may be inadequate. A learning curve does not always show a textbook U-shape.

In [ ]:
def predict_nn(frame: pd.DataFrame) -> np.ndarray:
    features = frame.loc[:, FEATURES]
    tensor = torch.tensor(x_scaler.transform(features), dtype=torch.float32, device=DEVICE)
    model.eval()
    with torch.no_grad():
        standardized = model(tensor).cpu().numpy().reshape(-1)
    return standardized * y_scale + y_mean

predictors = {name: estimator.predict for name, estimator in reference_models.items()}
predictors["Neural network"] = predict_nn
validation_results = pd.DataFrame([
    {"model": name, "train_RMSE_MPa": metrics(y_train, predict(X_train))["RMSE_MPa"],
     **metrics(y_val, predict(X_val))}
    for name, predict in predictors.items()
]).set_index("model")
selected_model_name = validation_results["RMSE_MPa"].idxmin()
display(validation_results.round(3))
print("Selected by validation RMSE:", selected_model_name)
print("No test scores were used in this selection.")

## 11 · Student experiments: change one idea, not the test set

For each experiment, first write a prediction. Then change the settings in Section 7, rerun Sections 7–10, and record the **training and validation** results. Do not keep rerunning the final test cell to pick your favorite model.

| Experiment | Suggested change | What to explain |
|---|---|---|
| Learning rate | Compare `0.0001`, `0.001`, and `0.01` | Which learns faster? Does fastest imply lowest validation error? |
| Model size | Compare hidden widths `(8, 8)` and `(128, 64)` | Which reduces training error most? Does validation improve too? |
| Batch size | Compare `16` and `128` | How many updates occur in one epoch? Are equal epochs equal numbers of updates? |
| Regularization | Compare weight decay `0` and `0.01` | How does the training/validation gap change? |

Use the editable record below. Results are not prefilled because you should observe your own run.

**Optional feature-engineering exercise:** Create `log_age = np.log1p(age_days)` and/or `water_binder_ratio`, then compare a linear model with and without those features using only training and validation data. Do not change the original eight-input NN without also changing its input layer. These are hypotheses to test, not guaranteed improvements.

**Evaluation honesty:** This notebook's first Run-all demonstration reveals the test scores. Subsequent classroom trials are exploratory. For a new formal claim, lock choices before evaluating a fresh reserved dataset or a properly nested, group-aware evaluation.

In [ ]:
# Fill this table with the results of your controlled experiments.
experiment_log = pd.DataFrame(columns=[
    "change", "prediction_before_running", "train_RMSE_MPa",
    "validation_RMSE_MPa", "best_epoch", "interpretation"])
display(experiment_log)

## 12 · Which inputs does the forest use?

Permutation importance shuffles one input column in the **validation** set, keeping the fitted model fixed, and measures how much the prediction error increases. The horizontal axis below is the increase in validation **MAE in MPa**, not a neural-network weight or a concrete material coefficient. [9]

The bars are means over repeated shuffles; error bars show the shuffle-to-shuffle standard deviation, **not confidence intervals**. Correlated ingredients and unrealistic shuffled combinations complicate interpretation. Importance is model- and dataset-dependent and does **not** establish causality. Negative values can occur.

In [ ]:
forest = reference_models["Random forest"]
importance = permutation_importance(
    forest, X_val, y_val, scoring="neg_mean_absolute_error",
    n_repeats=8, random_state=MODEL_SEED, n_jobs=1)
importance_table = pd.DataFrame({
    "feature": FEATURES,
    "increase_in_validation_MAE_MPa": importance.importances_mean,
    "shuffle_std_MPa": importance.importances_std,
}).sort_values("increase_in_validation_MAE_MPa")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importance_table["feature"], importance_table["increase_in_validation_MAE_MPa"],
        xerr=importance_table["shuffle_std_MPa"])
ax.axvline(0, linestyle="--")
ax.set(xlabel="Increase in validation MAE after shuffling (MPa)",
       title="Random-forest permutation importance; not a causal ranking")
fig.tight_layout()
plt.show()

## 13 · Make a new prediction in physical units

Edit the fields below, then run the cell. The model receives the same eight features, in the same order and units used for training. Preprocessing is reused; it is **not refitted** on the new specimen.

We flag inputs outside the training feature ranges. Being inside every individual range does **not** guarantee that the combination is represented in training data, physically feasible, or safe. The model does not check workability, curing procedures, durability, or compliance with design standards. It also does not provide a calibrated uncertainty interval.

The default inputs below are an illustrative mixture, not a recommended design. Changing one ingredient independently may change the mixture volume or physical feasibility. Compare predictions, but do not treat model agreement as proof of correctness.

In [ ]:
#@title Enter ingredients in kg/m³ and age in days
cement = 300.0 #@param {type:"number"}
slag = 100.0 #@param {type:"number"}
fly_ash = 0.0 #@param {type:"number"}
water = 180.0 #@param {type:"number"}
superplasticizer = 10.0 #@param {type:"number"}
coarse_aggregate = 950.0 #@param {type:"number"}
fine_aggregate = 800.0 #@param {type:"number"}
age_days = 28.0 #@param {type:"number"}

new_values = np.array([cement, slag, fly_ash, water, superplasticizer,
                       coarse_aggregate, fine_aggregate, age_days], dtype=float)
if not np.isfinite(new_values).all() or (new_values < 0).any():
    raise ValueError("All inputs must be finite and nonnegative.")
if cement <= 0 or water <= 0 or coarse_aggregate <= 0 or fine_aggregate <= 0 or age_days < 1:
    raise ValueError("For this lesson, cement, water, and both aggregates must be positive, "
                     "and age must be at least one day.")
new_mix = pd.DataFrame([new_values], columns=FEATURES)

outside = (new_mix.iloc[0] < X_train.min()) | (new_mix.iloc[0] > X_train.max())
if outside.any():
    print("WARNING — outside training ranges:", ", ".join(outside.index[outside]))
else:
    print("Inputs lie inside individual training ranges; joint coverage is NOT guaranteed.")
new_predictions = pd.DataFrame([
    {"model": name, "predicted_strength_MPa": float(predict(new_mix)[0])}
    for name, predict in predictors.items()
]).set_index("model")
display(new_predictions.round(2))
print("Model selected using validation data:", selected_model_name)
if (new_predictions["predicted_strength_MPa"] < 0).any():
    print("WARNING — a model produced a nonphysical negative strength; it has not been clipped.")
print("Educational prediction only; not a design or acceptance decision.")

## 14 · Final held-out evaluation

**Freeze your choices before running this cell.** The selected model name was determined from validation error. The table reports all four predetermined models for teaching comparison; do not re-select the winner using this test table.

The prediction plot should follow the one-to-one line. The second plot shows residuals, defined here as **predicted minus measured strength**. Positive residuals mean overprediction. Large overpredictions are particularly worth investigating in an engineering context, but no design-safety threshold is inferred here.

A single grouped holdout split estimates performance on these reserved recipe families. It does not prove that a model will transfer to a different laboratory, region, material source, or experimental protocol.

In [ ]:
frozen_selected_name = selected_model_name
TEST_PREDICTIONS = {name: predict(X_test) for name, predict in predictors.items()}
test_results = pd.DataFrame([
    {"model": name, **metrics(y_test, prediction)}
    for name, prediction in TEST_PREDICTIONS.items()
]).set_index("model")
display(test_results.round(3))
print("Previously selected model (unchanged):", frozen_selected_name)

selected_prediction = TEST_PREDICTIONS[frozen_selected_name]
lo = min(y_test.min(), selected_prediction.min()) - 2
hi = max(y_test.max(), selected_prediction.max()) + 2
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, selected_prediction, s=24, alpha=0.65)
ax.plot([lo, hi], [lo, hi], linestyle="--", label="Perfect prediction")
ax.set(xlabel="Measured strength (MPa)", ylabel="Predicted strength (MPa)",
       title=f"Held-out recipe families: {frozen_selected_name}",
       xlim=(lo, hi), ylim=(lo, hi))
ax.set_aspect("equal", adjustable="box")
ax.legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(selected_prediction, selected_prediction - y_test, s=24, alpha=0.65)
ax.axhline(0, linestyle="--")
ax.set(xlabel="Predicted strength (MPa)", ylabel="Prediction − measurement (MPa)",
       title="Test residuals: look for bias and large errors")
fig.tight_layout()
plt.show()

## 15 · Save a reproducible record and explain what you learned

The last cell saves the split assignments, settings, library versions, learning history, evaluation tables, and test predictions under `outputs/`. In Colab, use the left-hand Files panel to save files you need before the runtime is deleted.

**Short submission:** Include the validation/test comparison, one learning-curve plot, one controlled hyperparameter experiment, and a paragraph explaining a failure mode. Answer: What was learned? When were weights updated? Why were recipe families kept together? Why is $R^2$ not “percent accuracy”? Why should this predictor not replace a concrete test?

**Takeaway:** The workflow is **define the prediction task → inspect and split data → fit preprocessing on training data → compare baselines → train and validate → freeze choices → test → examine engineering limitations**. A more complex model is useful only when the evidence supports it.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
validation_results.to_csv(output_dir / "validation_metrics.csv")
test_results.to_csv(output_dir / "test_metrics.csv")
history.to_csv(output_dir / "learning_history.csv", index=False)
importance_table.to_csv(output_dir / "validation_permutation_importance.csv", index=False)

assignment = np.full(len(df), "", dtype=object)
for label, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    assignment[idx] = label
pd.DataFrame({"uci_data_row_1based": source_row_ids, "recipe_family": family_id,
              "split": assignment}).to_csv(output_dir / "split_assignments.csv", index=False)

test_export = X_test.copy()
test_export.insert(0, "uci_data_row_1based", source_row_ids[test_idx])
test_export["measured_strength_MPa"] = y_test
for name, prediction in TEST_PREDICTIONS.items():
    test_export[name + "_prediction_MPa"] = prediction
test_export.to_csv(output_dir / "test_predictions.csv", index=False)

run_report = {
    "data_sha256": DATA_SHA256, "raw_rows": len(raw), "cleaned_rows": len(df),
    "recipe_tolerance_kg_m3": RECIPE_TOLERANCE, "recipe_families": int(n_families),
    "split_seed": SPLIT_SEED, "model_seed": MODEL_SEED,
    "split_sizes": {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)},
    "hidden_widths": [HIDDEN_1, HIDDEN_2], "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
    "weight_decay": WEIGHT_DECAY, "restored_epoch": best_epoch,
    "epochs_run": len(history), "optimizer_updates": update_count,
    "neural_training_seconds": training_seconds,
    "selected_on_validation": str(frozen_selected_name), "versions": VERSIONS,
    "validation_metrics": validation_results.to_dict(orient="index"),
    "test_metrics": test_results.to_dict(orient="index"),
}
(output_dir / "run_report.json").write_text(json.dumps(run_report, indent=2))
assert np.isfinite(test_results.to_numpy()).all()
assert all(len(p) == len(y_test) for p in TEST_PREDICTIONS.values())
print("Saved run files:")
for path in sorted(output_dir.iterdir()):
    print(" ", path)
print("Notebook completed successfully.")

## Sources and further reading

[1] UCI Machine Learning Repository. [Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength), I-Cheng Yeh; [dataset DOI](https://doi.org/10.24432/C5PK67); [public CSV](https://archive.ics.uci.edu/static/public/165/data.csv). Data license: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The original publication is Yeh (1998), *Modeling of strength of high-performance concrete using artificial neural networks*, Cement and Concrete Research, 28(12), 1797–1808.

[2] scikit-learn. [Common pitfalls: inconsistent preprocessing and data leakage](https://scikit-learn.org/stable/common_pitfalls.html).

[3] scikit-learn. [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html). The 1 kg/m³ grouping rule in this lesson is our explicit preprocessing choice, not a rule supplied by this documentation or by UCI.

[4] scikit-learn. [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html).

[5] scikit-learn. [Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

[6] PyTorch. [Optimizing model parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).

[7] PyTorch. [Build the neural network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).

[8] PyTorch. [Adam optimizer](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html).

[9] scikit-learn. [Permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html).

**Reproducibility note:** This is an instructional reimplementation, not an exact reproduction of the original paper. Scores depend on data handling, the split, hyperparameters, and software versions. Seeds aid repeatability but do not guarantee bitwise identity across all software and hardware configurations.